# ML-02 — Research Question and Provisional Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**Lane 2 — Refresh / Content Opportunity Scoring**, a core lane, not an advanced one. I'm here
because my Week-1 work already sits in this lane. The starter pipeline builds a ranked refresh
queue with reason codes and a Precision@K score, and in notebook 02 I compared a hand rule
against a small tree on exactly that task. So I already have a baseline, a metric, and a first
result to build on. Sharpening the question this pipeline answers is a better use of seven weeks
than starting a different lane cold.

In [1]:
from pathlib import Path
import pandas as pd, numpy as np

root = Path.cwd()
while not (root / "data" / "raw" / "content_refresh_anonymized.csv").exists():
    if root.parent == root:
        raise FileNotFoundError("repo root with data/raw/ not found")
    root = root.parent

df = pd.read_csv(root / "data" / "raw" / "content_refresh_anonymized.csv")

# one row = one content page. the lane orders the VISIBLE pages.
visible = df[df["impressions_90d"] >= 100]
print(f"pages: {len(df):,}    visible (impressions_90d >= 100): {len(visible):,}")


pages: 30,000    visible (impressions_90d >= 100): 22,006


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

The question: **among pages already visible in search, which ones should an editor review first
for a refresh?** The decision it improves is the order of an editor's weekly worklist. The person
acting is a content editor who can realistically work a handful of pages a week; they open the top
of the queue and refresh, expand, protect, or keep watching based on the reason code.

A wrong call isn't free. Rank a page high when a refresh won't help and you burn scarce review
hours — worse, re-editing a stable page can lose rankings it already had. Bury a page that genuinely
needed attention and it declines quietly with nobody getting to it. Because the scarce resource is
attention, the ordering at the top of the list matters more than global accuracy.

In [2]:
# an editor reviews only a handful of pages a week, so the product is the ORDERING, not a flag.
weekly_capacity = 50
print(f"candidate visible pages: {len(visible):,}")
print(f"an editor realistically reviews ~{weekly_capacity}/week -> "
      f"{len(visible)/weekly_capacity:,.0f} weeks to touch them all without prioritizing.")
print("=> what matters is which pages land in the first Precision@K, not global accuracy.")


candidate visible pages: 22,006
an editor realistically reviews ~50/week -> 440 weeks to touch them all without prioritizing.
=> what matters is which pages land in the first Precision@K, not global accuracy.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

Three numbers from the starter slice that say this lane is worth the time:

1. **Search volume barely predicts delivered traffic** — near-zero correlation. Intent and what a
   page actually gets are different things, so a refresh queue should key off delivered performance,
   not keyword volume.
2. **A large share of visible pages are trending down** — plenty of candidates, which is exactly why
   they need *prioritizing* rather than a yes/no flag.
3. **And the decline isn't on dead pages** — over half of all visible impressions sit on the declining
   ones. That's real traffic at risk, which is what makes the ordering worth an editor's seven weeks.

Numbers computed live below.

In [3]:
# 1) does keyword search volume predict delivered impressions?
corr = df["search_volume"].corr(df["impressions_90d"])

# 2) share of visible pages trending down (proxy = trend_direction, as the pipeline derives it)
down = visible["trend_direction"].str.lower().eq("down")
declining_share = down.mean()

# 3) how much delivered traffic sits on those declining pages? (value at risk, not dead pages)
imp_at_risk = visible.loc[down, "impressions_90d"].sum() / visible["impressions_90d"].sum()

print(f"1. corr(search_volume, impressions_90d) = {corr:.3f}   (near zero -> intent != delivered traffic)")
print(f"2. visible pages trending down                = {declining_share:.1%}")
print(f"3. visible impressions on those pages         = {imp_at_risk:.1%}   (real traffic at risk)")


1. corr(search_volume, impressions_90d) = 0.001   (near zero -> intent != delivered traffic)
2. visible pages trending down                = 59.8%
3. visible impressions on those pages         = 51.3%   (real traffic at risk)


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**What this work can say:** observed and directional things on this sample — *these signals line up
with decline here*, *this is a ranked queue to inform an editor*. Decision-support, with the base
rate and caveats next to every number.

**What it will never say:** that any signal *causes* ranking movement, that it is *predicting Google*,
or that a page *will* recover if refreshed. Everything runs on the anonymized starter slice; no client
names, URLs, or private queries enter the work.

In [4]:
# a small guardrail I will hold myself to when writing results
ok_to_say   = ["observed on this sample", "directional", "associated with", "decision-support"]
never_say   = ["causes", "proves the ranking factor", "predicting Google", "guaranteed to recover"]
print("claim language — allowed :", ok_to_say)
print("claim language — off-limits:", never_say)


claim language — allowed : ['observed on this sample', 'directional', 'associated with', 'decision-support']
claim language — off-limits: ['causes', 'proves the ranking factor', 'predicting Google', 'guaranteed to recover']


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.